# ART calculations for `Bacillus Anthracis` bacteria

### Training data base `Pubchem + chEMBL`

In [ ]:
import sys
import os

# Completely suppress stderr output
sys.stderr = open(os.devnull, 'w')

# Now import everything
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
sys.path.append('.')   # Add local directory to access some of the functions
sys.path.append('../../') # Make sure this is the location for the ART library

import warning_utils
warning_utils.filter_end_user_warnings()

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem

from art.core import RecommendationEngine
import art.utility as utils
import pickle
import cloudpickle
import matplotlib.pyplot as plt
import gc

### Define directories

In [3]:
dataDir = '/code/DTRA_ART/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')
resultsDir = os.path.join(dataDir, 'Results/')
artResultDir = resultsDir + 'ART_results/BacillusAnthracis_bestMACAW_allDB/'
os.makedirs(artResultDir, exist_ok=True)
#saveDir = os.path.join(resultsDir, "Bacillus_anthracis/")
saveDir = os.path.join(resultsDir, "Bacillus_anthracis_bestMACAW_allDB/")
os.makedirs(saveDir, exist_ok=True)

### Extract the data for `Bacillus Anthracis` into a data frame with `duplicate` SMILES

In [4]:
BacillusAnthracisData_allDB_wMACAW = pd.read_csv(modelBuildingDataDir + "BacillusAnthracisData_allDB_wMACAW.csv")
BacillusAnthracisData_allDB_wMACAW 

,ID,compound_id,Smiles,pPotency,BacteriaClassifier,canonical_smiles,MACAW_1,MACAW_2,MACAW_3,MACAW_4,...,MACAW_51,MACAW_52,MACAW_53,MACAW_54,MACAW_55,MACAW_56,MACAW_57,MACAW_58,MACAW_59,MACAW_60
0,1,76310067,O=C([O-])[C@H](Cc1ccccc1)N1C(=O)/C(=C\c2ccc(-c...,4.086186,Bacillus_anthracis,O=C([O-])[C@H](Cc1ccccc1)N1C(=O)/C(=C\c2ccc(-c...,-0.096161,0.033215,-0.021804,0.077475,...,0.012225,0.005391,0.001515,-0.051672,0.016327,0.025795,0.008398,-0.007190,0.053421,0.003548
1,2,73055013,Cc1cc(N)c2cc(NC(=O)c3ccc(-c4ccc5ncccc5c4)cc3)c...,5.943095,Bacillus_anthracis,Cc1cc(N)c2cc(NC(=O)c3ccc(-c4ccc5ncccc5c4)cc3)c...,-0.166086,0.180249,0.017045,-0.010777,...,-0.048949,0.027629,-0.068264,-0.037689,0.011902,-0.083724,-0.047881,0.006992,-0.005103,0.003730
2,3,73055018,COc1ccccc1CCC(=O)Nc1ccc2nc(C)cc(N)c2c1,5.540608,Bacillus_anthracis,COc1ccccc1CCC(=O)Nc1ccc2nc(C)cc(N)c2c1,-0.099916,0.153509,-0.059641,0.046942,...,-0.080056,-0.056573,0.085205,-0.025114,0.006599,0.014874,-0.001325,-0.063239,0.031737,-0.058246
3,4,73055013,Cc1cc(N)c2cc(NC(=O)c3ccc(-c4ccc5ncccc5c4)cc3)c...,5.957818,Bacillus_anthracis,Cc1cc(N)c2cc(NC(=O)c3ccc(-c4ccc5ncccc5c4)cc3)c...,-0.166086,0.180249,0.017045,-0.010777,...,-0.048949,0.027629,-0.068264,-0.037689,0.011902,-0.083724,-0.047881,0.006992,-0.005103,0.003730
4,5,73055018,COc1ccccc1CCC(=O)Nc1ccc2nc(C)cc(N)c2c1,5.913996,Bacillus_anthracis,COc1ccccc1CCC(=O)Nc1ccc2nc(C)cc(N)c2c1,-0.099916,0.153509,-0.059641,0.046942,...,-0.080056,-0.056573,0.085205,-0.025114,0.006599,0.014874,-0.001325,-0.063239,0.031737,-0.058246
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4076,4331,CHEMBL560502,CCCC1c2ccccc2C=NN1C(=O)/C=C/c1cc(Cc2cnc(N)nc2N...,7.267606,Bacillus_anthracis,CCCC1c2ccccc2C=NN1C(=O)/C=C/c1cc(Cc2cnc(N)nc2N...,-0.217627,0.019361,0.084149,0.144391,...,0.052784,-0.005557,0.109101,0.030033,0.042055,-0.008807,0.010615,0.009777,-0.004671,0.054411
4077,4333,CHEMBL561523,COc1cc(/C=C2\SC(=S)N(c3cccc(C(F)(F)F)c3)C2=O)c...,4.022276,Bacillus_anthracis,COc1cc(/C=C2\SC(=S)N(c3cccc(C(F)(F)F)c3)C2=O)c...,-0.019458,0.070758,0.025811,0.050822,...,0.006941,-0.026578,0.039326,0.038939,-0.067496,-0.050675,0.001596,-0.058739,-0.076435,0.017447
4078,4334,CHEMBL560919,Cc1c(Cc2ccccc2)c(=O)oc2c(O)c(O)ccc12,5.000000,Bacillus_anthracis,Cc1c(Cc2ccccc2)c(=O)oc2c(O)c(O)ccc12,0.079721,0.123650,-0.050064,0.026177,...,-0.042918,-0.039452,-0.012660,0.001914,0.075364,-0.033146,0.007812,0.054100,-0.054057,-0.047724
4079,4337,CHEMBL5191499,O=C1CC(c2ccccc2)=NN1c1nc2ccccc2[nH]c1=O,7.800000,Bacillus_anthracis,O=C1CC(c2ccccc2)=NN1c1nc2ccccc2[nH]c1=O,-0.004541,0.192132,-0.016756,-0.034928,...,0.007025,-0.005243,-0.065841,-0.001203,0.028444,0.020824,-0.043441,-0.053132,0.060188,0.005175


In [5]:
print("Shape before cleaning pPotency data (3 < pPotency < 10):", BacillusAnthracisData_allDB_wMACAW.shape)

BacillusAnthracisData_allDB_wMACAW = BacillusAnthracisData_allDB_wMACAW[
    BacillusAnthracisData_allDB_wMACAW["pPotency"].between(3.0, 10.0)
]

print("Shape after cleaning pPotency data (3 < pPotency < 10):", BacillusAnthracisData_allDB_wMACAW.shape)

Shape before cleaning pPotency data (3 < pPotency < 10): (4081, 66)
Shape after cleaning pPotency data (3 < pPotency < 10): (3260, 66)


### Prepare data to run `ART` on `Bacillus Anthracis` data with `duplicate` SMILES

#### Find Features and Response

In [6]:
input_var = [col for col in BacillusAnthracisData_allDB_wMACAW.columns if col.startswith('MACAW_')]
print(f"MACAW Embeddings: {len(input_var)}")
print(input_var)

MACAW Embeddings: 60
['MACAW_1', 'MACAW_2', 'MACAW_3', 'MACAW_4', 'MACAW_5', 'MACAW_6', 'MACAW_7', 'MACAW_8', 'MACAW_9', 'MACAW_10', 'MACAW_11', 'MACAW_12', 'MACAW_13', 'MACAW_14', 'MACAW_15', 'MACAW_16', 'MACAW_17', 'MACAW_18', 'MACAW_19', 'MACAW_20', 'MACAW_21', 'MACAW_22', 'MACAW_23', 'MACAW_24', 'MACAW_25', 'MACAW_26', 'MACAW_27', 'MACAW_28', 'MACAW_29', 'MACAW_30', 'MACAW_31', 'MACAW_32', 'MACAW_33', 'MACAW_34', 'MACAW_35', 'MACAW_36', 'MACAW_37', 'MACAW_38', 'MACAW_39', 'MACAW_40', 'MACAW_41', 'MACAW_42', 'MACAW_43', 'MACAW_44', 'MACAW_45', 'MACAW_46', 'MACAW_47', 'MACAW_48', 'MACAW_49', 'MACAW_50', 'MACAW_51', 'MACAW_52', 'MACAW_53', 'MACAW_54', 'MACAW_55', 'MACAW_56', 'MACAW_57', 'MACAW_58', 'MACAW_59', 'MACAW_60']


In [7]:
features = BacillusAnthracisData_allDB_wMACAW[input_var].to_numpy()

In [8]:
response_var = ["pPotency"]
print(response_var)

['pPotency']


In [9]:
response = BacillusAnthracisData_allDB_wMACAW[response_var].to_numpy()

### save the data as a EDD style file

In [10]:
utils.save_edd_csv(features, response, input_var, modelBuildingDataDir + 'BacillusAnthracisData_allDB_wMACAW_ARTready.csv', response_var)

### Predict response with ART

In [11]:
BacillusAnthracisData_allDB_wMACAW_ARTready = pd.read_csv(modelBuildingDataDir + "BacillusAnthracisData_allDB_wMACAW_ARTready.csv")
BacillusAnthracisData_allDB_wMACAW_ARTready

,Line Name,Type,0.0
0,0,MACAW_1,-0.096161
1,1,MACAW_1,-0.166086
2,2,MACAW_1,-0.099916
3,3,MACAW_1,-0.166086
4,4,MACAW_1,-0.099916
...,...,...,...
198855,3255,pPotency,6.000000
198856,3256,pPotency,7.267606
198857,3257,pPotency,4.022276
198858,3258,pPotency,5.000000


### Define the ART parameters needed for the prediction

In [12]:
art_params = {
    'input_vars': input_var,
    'response_vars': response_var,
    'objective': 'maximize',
    'threshold': 0.2,
    'alpha': 0.5,
    'num_recommendations': 10,
    'max_mcmc_cores': 4,
    'seed': 42,                    
    'output_dir': artResultDir,
    'recommend': False,
    'cross_val': True,
    'num_tpot_models': 2,
}

### Run ART without recommendations but with cross-validations to gauge how generalizable the results are

In [ ]:
%%time

art = RecommendationEngine(df=BacillusAnthracisData_allDB_wMACAW_ARTready, **art_params)

ART identified 3,260 unique designs in the training data. 
Correctly communicating your designs is critical to building ART's model and to cross-validation.  We strongly suggest verifying ART's interpretation of your designs by inspecting the preprocessed DataFrame.  For example, if your ART instance is named `engine`, run `engine.df["Design"]`.  See Data Preparation docs for a description of how ART infers design information. https://lbl-biosci.gitlab.io/ese/art/Data_Preparation/#labeling-lines-designs-replicates-oh-my
is_classifier
is_classifier
is_classifier
is_classifier
is_regressor
is_regressor
is_regressor
is_regressor
is_classifier
is_classifier
is_classifier
is_classifier
is_regressor
is_regressor
is_regressor
is_regressor
is_classifier
is_classifier
is_classifier
is_classifier
is_regressor
is_regressor
is_regressor
is_regressor
is_classifier
is_classifier
is_classifier
is_classifier
is_regressor
is_regressor
is_regressor
is_regressor
is_classifier
is_classifier
is_classifier


### Find SHAP values

We will now find which input features are more important by using SHAP analysis. First, lets initialize the library:

define a wrapper function that provides the ART prediction given and input X, for use by the SHAP library

convert the ART input data into the pandas dataframe that the SHAP library favors

create and execute the explainer for the features values

### Load pre-trained ART model (if necessary, otherwise skip this step)

In [ ]:
ARTtrainedModelFile = os.path.join(artResultDir, 'art.cpkl')
with open(ARTtrainedModelFile, 'rb') as f:
    art = cloudpickle.load(f)
print(f" ART model loaded from: {ARTtrainedModelFile}")

### Load MACAW transformer (not required at this moment)

### Find external validation with pre-trained `ART` models

In [ ]:
BacillusAnthracis_all_Buyable_AntiVirals_wMACAW = pd.read_csv(modelBuildingDataDir + "/BacillusAnthracis_all_Buyable_Antibacterials_wMACAW.csv")

print(f"Total buyable antiviral molecules before removing training set molecules: {len(BacillusAnthracis_all_Buyable_AntiVirals_wMACAW)}")

# Get the set of training SMILES from the chEMBL data for fast lookup
training_smiles_set = set(BacillusAnthracisData_allDB_wMACAW['Smiles'])

# Remove rows whose SMILES are present in the training data
BacillusAnthracis_all_Buyable_AntiVirals_wMACAW = BacillusAnthracis_all_Buyable_AntiVirals_wMACAW[
    ~BacillusAnthracis_all_Buyable_AntiVirals_wMACAW['SMILES'].isin(training_smiles_set)
].reset_index(drop=True)

removed_count = len(training_smiles_set) - (len(training_smiles_set) - (len(BacillusAnthracis_all_Buyable_AntiVirals_wMACAW)))
print(f"Training set molecules found and removed: {len(pd.read_csv(modelBuildingDataDir + '/Ebola_all_Buyable_AntiVirals_wMACAW.csv')) - len(BacillusAnthracis_all_Buyable_AntiVirals_wMACAW)}")
print(f"Total buyable antiviral molecules for external validation after removing training set molecules: {len(BacillusAnthracis_all_Buyable_AntiVirals_wMACAW)}")

BacillusAnthracis_all_Buyable_AntiVirals_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_all_Buyable_AntiVirals_wMACAW.columns if col.startswith('MACAW_')]
smiLib = BacillusAnthracis_all_Buyable_AntiVirals_wMACAW[macaw_columns].values

### Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smiLib)

# Now use these for your results
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction = BacillusAnthracis_all_Buyable_AntiVirals_wMACAW[['SMILES']].copy()
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['pPotency_prediction'] = mean
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['pPotency_std'] = std

# Calculate 95% Confidence Intervals
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['pPotency_lower_95CI'] = mean - 1.96 * std
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['IC50(M)_prediction'] = 10 ** (-mean)
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['IC50(M)_lower_95CI'] = 10 ** (-BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['pPotency_upper_95CI'])
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['IC50(M)_upper_95CI'] = 10 ** (-BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction['pPotency_lower_95CI'])

# Select and save results
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction = BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction.to_csv(os.path.join(saveDir + "/BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction.csv"), index=False)
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction[['SMILES']].to_csv(os.path.join(saveDir + "/BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
BacillusAnthracis_all_Buyable_AntiVirals_ARTprediction

## `DORAnet Generated Antibiotic:` Find external validation with pre-trained `ART` models

In [ ]:
def canonicalize(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol) if mol else smi

# Load DORAnet generated molecules
BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW = pd.read_csv(
    modelBuildingDataDir + "/BacillusAnthracisData_allDB_basidalinPrecursor_DORAnetGenerated_Antibiotic_wMACAW.csv")

total_before = len(BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW)
print(f"Total DORAnet generated molecules before removing training set molecules: {total_before}")

# Training set stats
total_training_rows = len(BacillusAnthracisData_allDB_wMACAW)
unique_training_smiles = BacillusAnthracisData_allDB_wMACAW['Smiles'].nunique()
duplicate_training_smiles = BacillusAnthracisData_allDB_wMACAW['Smiles'].duplicated().sum()
print(f"Total training set SMILES: {total_training_rows}")
print(f"Unique training set SMILES: {unique_training_smiles}")
print(f"Duplicate training set SMILES: {duplicate_training_smiles}")

# Canonicalize training SMILES for consistent matching
training_smiles_set = set(BacillusAnthracisData_allDB_wMACAW['Smiles'].apply(canonicalize))
print(f"Unique canonical training SMILES: {len(training_smiles_set)}")

# Canonicalize DORAnet SMILES and remove matches
BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW['canonical_SMILES'] = \
    BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW['SMILES'].apply(canonicalize)

BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW = \
    BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW[
        ~BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW['canonical_SMILES'].isin(training_smiles_set)
    ].drop(columns=['canonical_SMILES']).reset_index(drop=True)

total_after = len(BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW)
print(f"\nTraining set molecules found and removed: {total_before - total_after}")
print(f"Total DORAnet generated molecules for external validation: {total_after}")

BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW.columns if col.startswith('MACAW_')]
smiLib = BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW[macaw_columns].values

### Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
chunk_size = 100000
total_rows = len(BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW)
print(f"Total molecules to predict: {total_rows}")
print(f"Processing in chunks of {chunk_size}...")

macaw_cols = [col for col in BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW.columns if col.startswith('MACAW_')]

output_path = os.path.join(saveDir, "BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_ARTprediction.csv")
smiles_output_path = os.path.join(saveDir, "BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_ARTprediction_SMILESonly.csv")

if os.path.exists(output_path):
    os.remove(output_path)
if os.path.exists(smiles_output_path):
    os.remove(smiles_output_path)

header_written = False

for start in range(0, total_rows, chunk_size):
    end = min(start + chunk_size, total_rows)
    print(f"  Processing chunk {start} - {end} ({end}/{total_rows})...")

    chunk_features = BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW.iloc[start:end][macaw_cols].values
    chunk_smiles = BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW.iloc[start:end]['SMILES'].values
    chunk_is_starter = BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW.iloc[start:end]['Is_Starter'].values
    chunk_source_dirs = BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_wMACAW.iloc[start:end]['source_DataBase'].values

    mean, std = art.post_pred_stats(chunk_features)
    mean = np.array(mean).flatten()
    std = np.array(std).flatten()

    chunk_df = pd.DataFrame({
        'SMILES': chunk_smiles,
        'Is_Starter': chunk_is_starter,
        'SourceDirectories': chunk_source_dirs,
        'pPotency_prediction': mean,
        'pPotency_std': std,
        'pPotency_lower_95CI': mean - 1.96 * std,
        'pPotency_upper_95CI': mean + 1.96 * std,
        'IC50(M)_prediction': 10 ** (-mean),
        'IC50(M)_lower_95CI': 10 ** (-(mean + 1.96 * std)),
        'IC50(M)_upper_95CI': 10 ** (-(mean - 1.96 * std)),
    })

    chunk_df.to_csv(output_path, mode='a', header=not header_written, index=False)
    chunk_df[['SMILES']].to_csv(smiles_output_path, mode='a', header=not header_written, index=False)
    header_written = True

    del chunk_features, chunk_smiles, chunk_is_starter, chunk_source_dirs, mean, std, chunk_df
    gc.collect()

print(f"\nPredictions saved to: {output_path}")
print(f"SMILES-only saved to: {smiles_output_path}")

BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_ARTprediction = pd.read_csv(output_path)
print(f"Total predictions: {len(BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_ARTprediction)}")
BacillusAnthracis_BasidalinPrecursorDORAnetGenerated_Antibiotic_ARTprediction.head()

## 2.1 Discovery of new hits specific to all bacteria (data source Enamine_AntiBioticData.csv)

In this section, we screen a custom virtual library looking for molecules that are promising accoring to the the SVR models `regr` above, which use 15-D MACAW embeddings as their input. The custom library ("Enamine_AntiBioticData.csv") compiled from commercial catalogs by Enamine. In particular, we are interested in molecules with high predicted pPotency.

Load MACAW embeedings for external validation data set

In [ ]:
BacillusAnthracis_EnamineDatasets_wMACAW = pd.read_csv(saveDir + "BacillusAnthracis_EnamineDataset_wMACAW.csv")
BacillusAnthracis_EnamineDatasets_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_EnamineDatasets_wMACAW.columns if col.startswith('MACAW_')]
smi_lib_EnamineDatasets_wMACAW = BacillusAnthracis_EnamineDatasets_wMACAW[macaw_columns].values

In [ ]:
Y1_lib_pred = art.predict(smi_lib_EnamineDatasets_wMACAW)
Y1_lib_pred

Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smi_lib_EnamineDatasets_wMACAW)

# Now use these for your results
EnamineAntiBioticData_predicted = BacillusAnthracis_EnamineDatasets_wMACAW[['SMILES']].copy()
EnamineAntiBioticData_predicted['pPotency_prediction'] = mean
EnamineAntiBioticData_predicted['pPotency_std'] = std

# Calculate 95% Confidence Intervals
EnamineAntiBioticData_predicted['pPotency_lower_95CI'] = mean - 1.96 * std
EnamineAntiBioticData_predicted['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
EnamineAntiBioticData_predicted['IC50(M)_prediction'] = 10 ** (-mean)
EnamineAntiBioticData_predicted['IC50(M)_lower_95CI'] = 10 ** (-EnamineAntiBioticData_predicted['pPotency_upper_95CI'])
EnamineAntiBioticData_predicted['IC50(M)_upper_95CI'] = 10 ** (-EnamineAntiBioticData_predicted['pPotency_lower_95CI'])

# Select and save results
EnamineAntiBioticData_predicted = EnamineAntiBioticData_predicted.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

EnamineAntiBioticData_predicted.to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_EnamineDataset_predicted_all.csv"), index=False)
EnamineAntiBioticData_predicted[['SMILES']].to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_EnamineDataset_predicted_all_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
EnamineAntiBioticData_predicted

## 2.2 Discovery of new hits specific to all bacteria (data source Life chemicals data set)

In this section, we screen a custom virtual library looking for molecules that are promising accoring to the the SVR models `regr` above, which use 15-D MACAW embeddings as their input. The custom library ("Enamine_AntiBioticData.csv") compiled from commercial catalogs by Enamine. In particular, we are interested in molecules with high predicted pPotency.

In [ ]:
BacillusAnthracis_LCAntiBioticData_wMACAW = pd.read_csv(saveDir + "BacillusAnthracis_LCAntiviralsData_wMACAW.csv")
BacillusAnthracis_LCAntiBioticData_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_LCAntiBioticData_wMACAW.columns if col.startswith('MACAW_')]
smi_lib_LCAntiBioticData_wMACAW = BacillusAnthracis_LCAntiBioticData_wMACAW[macaw_columns].values

In [ ]:
Y1_lib_pred = art.predict(smi_lib_LCAntiBioticData_wMACAW)
Y1_lib_pred

Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smi_lib_LCAntiBioticData_wMACAW)

# Now use these for your results
LCAntiBioticData_predicted = BacillusAnthracis_LCAntiBioticData_wMACAW[['SMILES']].copy()
LCAntiBioticData_predicted['pPotency_prediction'] = mean
LCAntiBioticData_predicted['pPotency_std'] = std

# Calculate 95% Confidence Intervals
LCAntiBioticData_predicted['pPotency_lower_95CI'] = mean - 1.96 * std
LCAntiBioticData_predicted['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
LCAntiBioticData_predicted['IC50(M)_prediction'] = 10 ** (-mean)
LCAntiBioticData_predicted['IC50(M)_lower_95CI'] = 10 ** (-LCAntiBioticData_predicted['pPotency_upper_95CI'])
LCAntiBioticData_predicted['IC50(M)_upper_95CI'] = 10 ** (-LCAntiBioticData_predicted['pPotency_lower_95CI'])

# Select and save results
LCAntiBioticData_predicted = LCAntiBioticData_predicted.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

LCAntiBioticData_predicted.to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_LCAntiBioticData_predicted_all.csv"), index=False)
LCAntiBioticData_predicted[['SMILES']].to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_LCAntiBioticData_predicted_all_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
LCAntiBioticData_predicted

## 2.3 Discovery of new hits specific to all viruses (data source `chemDiv` data set)

In this section, we screen a custom virtual library looking for molecules that are promising accoring to the the SVR models `regr` above, which use 15-D MACAW embeddings as their input. The custom library compiled from commercial catalogs by Enamine. In particular, we are interested in molecules with high predicted pPotency.

In [ ]:
BacillusAnthracis_ChemDivAntiBioticData_wMACAW = pd.read_csv(saveDir + "BacillusAnthracis_ChemDivAntiviralsData_wMACAW.csv")
BacillusAnthracis_ChemDivAntiBioticData_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_ChemDivAntiBioticData_wMACAW.columns if col.startswith('MACAW_')]
smi_lib_ChemDivAntiBioticData_wMACAW = BacillusAnthracis_ChemDivAntiBioticData_wMACAW[macaw_columns].values

In [ ]:
Y1_lib_pred = art.predict(smi_lib_ChemDivAntiBioticData_wMACAW)
Y1_lib_pred

Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smi_lib_ChemDivAntiBioticData_wMACAW)

# Now use these for your results
ChemDivAntiBioticData_predicted = BacillusAnthracis_ChemDivAntiBioticData_wMACAW[['SMILES']].copy()
ChemDivAntiBioticData_predicted['pPotency_prediction'] = mean
ChemDivAntiBioticData_predicted['pPotency_std'] = std

# Calculate 95% Confidence Intervals
ChemDivAntiBioticData_predicted['pPotency_lower_95CI'] = mean - 1.96 * std
ChemDivAntiBioticData_predicted['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
ChemDivAntiBioticData_predicted['IC50(M)_prediction'] = 10 ** (-mean)
ChemDivAntiBioticData_predicted['IC50(M)_lower_95CI'] = 10 ** (-ChemDivAntiBioticData_predicted['pPotency_upper_95CI'])
ChemDivAntiBioticData_predicted['IC50(M)_upper_95CI'] = 10 ** (-ChemDivAntiBioticData_predicted['pPotency_lower_95CI'])

# Select and save results
ChemDivAntiBioticData_predicted = ChemDivAntiBioticData_predicted.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

ChemDivAntiBioticData_predicted.to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_ChemDivAntiBioticData_predicted_all.csv"), index=False)
ChemDivAntiBioticData_predicted[['SMILES']].to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_ChemDivAntiBioticData_predicted_all_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
ChemDivAntiBioticData_predicted

### 2.4 Checking drug likeliness of `Basidalin`

In [ ]:
BacillusAnthracis_DTRA_target_AntiBiotic_wMACAW = pd.read_csv(saveDir + "BacillusAnthracis_DTRA_smiles_wMACAW.csv")
BacillusAnthracis_DTRA_target_AntiBiotic_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_DTRA_target_AntiBiotic_wMACAW.columns if col.startswith('MACAW_')]
smi_lib_DTRA_target_AntiBiotic_wMACAW = BacillusAnthracis_DTRA_target_AntiBiotic_wMACAW[macaw_columns].values

In [ ]:
Y1_lib_pred = art.predict(smi_lib_DTRA_target_AntiBiotic_wMACAW)
Y1_lib_pred

Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smi_lib_DTRA_target_AntiBiotic_wMACAW)

# Now use these for your results
DTRA_target_AntiBiotic_predicted = BacillusAnthracis_DTRA_target_AntiBiotic_wMACAW[['SMILES']].copy()
DTRA_target_AntiBiotic_predicted['pPotency_prediction'] = mean
DTRA_target_AntiBiotic_predicted['pPotency_std'] = std

# Calculate 95% Confidence Intervals
DTRA_target_AntiBiotic_predicted['pPotency_lower_95CI'] = mean - 1.96 * std
DTRA_target_AntiBiotic_predicted['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
DTRA_target_AntiBiotic_predicted['IC50(M)_prediction'] = 10 ** (-mean)
DTRA_target_AntiBiotic_predicted['IC50(M)_lower_95CI'] = 10 ** (-DTRA_target_AntiBiotic_predicted['pPotency_upper_95CI'])
DTRA_target_AntiBiotic_predicted['IC50(M)_upper_95CI'] = 10 ** (-DTRA_target_AntiBiotic_predicted['pPotency_lower_95CI'])

# Select and save results
DTRA_target_AntiBiotic_predicted = DTRA_target_AntiBiotic_predicted.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

DTRA_target_AntiBiotic_predicted.to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_DTRA_target_AntiBiotic_predicted_all.csv"), index=False)
DTRA_target_AntiBiotic_predicted[['SMILES']].to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_DTRA_target_AntiBiotic_predicted_all_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
DTRA_target_AntiBiotic_predicted

## 2.5 Discovery of new hits specific to all Bacteriaes (data source `Retrotide` generated molecules)

In this section, we screen a custom virtual library looking for molecules that are promising accoring to the the SVR models `regr` above, which use 15-D MACAW embeddings as their input. The library of compounds has been generated using `Retrotide`. In particular, we are interested in molecules with high predicted pPotency. Here is the settings and corresponding notebook link:

0. Start from first PKS compound of Basidalin synthesis
1. Keep fixed module1
2. diversify only starter and second extension module
3. The TE domain is trying to cyclize the molecule, but the chemical reaction pattern doesn't match the molecule structure, so rxn.RunReactants() returns an empty result with 73.6% failure rate. So we are using different ring positionsto find most diverse set of molecules.

https://github.com/soumyadeepghosh35/RetroTide/blob/DTRA_Retrotide/notebooks/Basidalin_Retrotide.ipynb

Load MACAW embeedings for external validation data set

In [ ]:
BacillusAnthracis_RetrotideDatasets_wMACAW = pd.read_csv(saveDir + "BacillusAnthracis_RetrotideDataset_wMACAW.csv")
BacillusAnthracis_RetrotideDatasets_wMACAW

In [ ]:
macaw_columns = [col for col in BacillusAnthracis_RetrotideDatasets_wMACAW.columns if col.startswith('MACAW_')]
smi_lib_RetrotideDatasets_wMACAW = BacillusAnthracis_RetrotideDatasets_wMACAW[macaw_columns].values

In [ ]:
Y1_lib_pred = art.predict(smi_lib_RetrotideDatasets_wMACAW)
Y1_lib_pred

Get predictions with uncertainty using ART's post_pred_stats

In [ ]:
# Pass X1_lib (features)
mean, std = art.post_pred_stats(smi_lib_RetrotideDatasets_wMACAW)

# Now use these for your results
RetrotideAntiBioticData_predicted = BacillusAnthracis_RetrotideDatasets_wMACAW[['SMILES']].copy()
RetrotideAntiBioticData_predicted['pPotency_prediction'] = mean
RetrotideAntiBioticData_predicted['pPotency_std'] = std

# Calculate 95% Confidence Intervals
RetrotideAntiBioticData_predicted['pPotency_lower_95CI'] = mean - 1.96 * std
RetrotideAntiBioticData_predicted['pPotency_upper_95CI'] = mean + 1.96 * std

# Convert to IC50
RetrotideAntiBioticData_predicted['IC50(M)_prediction'] = 10 ** (-mean)
RetrotideAntiBioticData_predicted['IC50(M)_lower_95CI'] = 10 ** (-RetrotideAntiBioticData_predicted['pPotency_upper_95CI'])
RetrotideAntiBioticData_predicted['IC50(M)_upper_95CI'] = 10 ** (-RetrotideAntiBioticData_predicted['pPotency_lower_95CI'])

# Select and save results
RetrotideAntiBioticData_predicted = RetrotideAntiBioticData_predicted.filter(
    items=["SMILES", "pPotency_prediction", "pPotency_std", 
           "pPotency_lower_95CI", "pPotency_upper_95CI",
           "IC50(M)_prediction", "IC50(M)_lower_95CI", "IC50(M)_upper_95CI"]
)

RetrotideAntiBioticData_predicted.to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_RetrotideDataset_predicted_all.csv"), index=False)
RetrotideAntiBioticData_predicted[['SMILES']].to_csv(os.path.join(saveDir + "BacillusAnthracis_wART_RetrotideDataset_predicted_all_SMILES.csv"), index=False)
print(f"Predictions saved with uncertainty estimates")
RetrotideAntiBioticData_predicted

### Top 20 compounds with higher pPotency

In [ ]:
# Sort by pPotency and select the top 20
RetrotideAntiBioticData_predicted_top20 = RetrotideAntiBioticData_predicted.sort_values(
    by='pPotency_prediction', ascending=False)

# Print the range of predictions
print("Top 20 Compounds:")
print(f"Range of pPotency_prediction: {RetrotideAntiBioticData_predicted_top20['pPotency_prediction'].min():.2f} - {RetrotideAntiBioticData_predicted_top20['pPotency_prediction'].max():.2f}")
print(f"Range of pPotency_std: {RetrotideAntiBioticData_predicted_top20['pPotency_std'].min():.2f} - {RetrotideAntiBioticData_predicted_top20['pPotency_std'].max():.2f}")
print("Range of IC50(M)_prediction:", RetrotideAntiBioticData_predicted_top20['IC50(M)_prediction'].min(), 
      "-", RetrotideAntiBioticData_predicted_top20['IC50(M)_prediction'].max())

RetrotideAntiBioticData_predicted_top20

# Prepare to run `ART` on `Bacillus Anthracis` data without `duplicate` SMILES

In [ ]:
BacillusAnthracisData_allDB_noDuplicates = pd.read_csv(modelBuildingDataDir + "BacillusAnthracisData_allDB_noDuplicates_wMACAW.csv")
BacillusAnthracisData_allDB_noDuplicates 

### Prepare data to run `ART` on `Bacillus Anthracis` data with `duplicate` SMILES

#### Find Features and Response

In [ ]:
input_var = [col for col in BacillusAnthracisData_allDB_noDuplicates.columns if col.startswith('MACAW_')]
print(f"MACAW Embeddings: {len(input_var)}")
print(input_var)

In [ ]:
features = BacillusAnthracisData_allDB_noDuplicates[input_var].to_numpy()

In [ ]:
response_var = ["pPotency"]
print(response_var)

In [ ]:
response = BacillusAnthracisData_allDB_noDuplicates[response_var].to_numpy()

### save the data as a EDD style file

In [ ]:
utils.save_edd_csv(features, response, input_var, modelBuildingDataDir + 'BacillusAnthracisData_allDB_noDuplicates_ARTready.csv', response_var)

### Predict response with ART

In [ ]:
BacillusAnthracisData_allDB_noDuplicates_ARTready = pd.read_csv(modelBuildingDataDir + "BacillusAnthracisData_allDB_noDuplicates_ARTready.csv")
BacillusAnthracisData_allDB_noDuplicates_ARTready

### Define the ART parameters needed for the prediction

In [ ]:
art_params = {
    'input_vars': input_var,
    'response_vars': response_var,
    'objective': 'maximize',
    'threshold': 0.2,
    'alpha': 0.5,
    'num_recommendations': 10,
    'max_mcmc_cores': 4,
    'seed': 42,                    
    'output_dir': artResultDir,
    'recommend': False,
    'cross_val': True,
    'num_tpot_models': 2,
}

### Run ART without recommendations but with cross-validations to gauge how generalizable the results are

In [ ]:
%%time

art = RecommendationEngine(df=BacillusAnthracisData_allDB_noDuplicates_ARTready, **art_params)